In [30]:
import cv2
import numpy as np
import time
from tqdm import tqdm

In [31]:
# SET PATHS SEPARATELY
VIDEO_PATH = "../Datasets/Processed_Data/Front/W001/W001S01F_01.mp4"         # change this
LANDMARK_PATH = "../Datasets/Landmarks/Front/W001/W001S01F_01.npy" 
OUTPUT_PATH = "../sample_output.mp4"  # ← OUTPUT FILE


In [32]:
# Load landmarks
landmarks = np.load(LANDMARK_PATH)
total_frames = landmarks.shape[0]

print("Loaded landmarks:", landmarks.shape)

Loaded landmarks: (60, 387)


In [33]:
# Open video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError("❌ ERROR: Cannot open video file")

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

In [34]:
# Setup video writer for output
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

print("Saving video to:", OUTPUT_PATH)

Saving video to: ../sample_output.mp4


In [35]:
def draw_points(frame, points, color):
    # Get image dimensions from the frame
    height, width, _ = frame.shape
    
    # Ensure points are processed if they are not empty
    if points.size == 0:
        return frame
        
    for point in points:
        # We only need the X and Y coordinates (the third value is usually Z)
        x_norm, y_norm = point[0], point[1]
        
        # Check if coordinates are valid numbers and within normalized range [0, 1]
        if np.isfinite(x_norm) and np.isfinite(y_norm) and 0 <= x_norm <= 1 and 0 <= y_norm <= 1:
            # Convert normalized coordinates to pixel values
            x = int(x_norm * width)
            y = int(y_norm * height)
            
            # Draw a filled circle at the landmark location
            cv2.circle(frame, (x, y), radius=2, color=color, thickness=-1)
            
    return frame


In [36]:
# Process video frame-by-frame
frame_index = 0

FACE_LM = 54     # face points
LH_LM = 21       # left-hand
RH_LM = 21       # right-hand
POSE_LM = 33     # pose

for i in tqdm(range(total_frames), desc="Rendering video"):

    ret, frame = cap.read()
    if not ret:
        break

    lm = landmarks[frame_index]

    # ----- Correct Slicing for 54 Face Landmarks -----
    idx = 0

    # Face: 54*3 values
    face = lm[idx : idx + FACE_LM*3].reshape(FACE_LM, 3)
    idx += FACE_LM*3

    # Left-hand: 21*3 values
    lh = lm[idx : idx + LH_LM*3].reshape(LH_LM, 3)
    idx += LH_LM*3

    # Right-hand: 21*3 values
    rh = lm[idx : idx + RH_LM*3].reshape(RH_LM, 3)
    idx += RH_LM*3

    # Pose: 33*3 values
    pose = lm[idx : idx + POSE_LM*3].reshape(POSE_LM, 3)

    # ----- Draw landmarks -----
    frame = draw_points(frame, face, (255, 0, 0))     # Blue → Face (54)
    frame = draw_points(frame, lh, (0, 255, 0))       # Green → Left Hand
    frame = draw_points(frame, rh, (0, 0, 255))       # Red → Right Hand
    frame = draw_points(frame, pose, (255, 255, 0))   # Yellow → Pose

    out.write(frame)
    frame_index += 1


cap.release()
out.release()

print("✅ Video saved successfully at:", OUTPUT_PATH)

Rendering video:  82%|████████▏ | 49/60 [00:00<00:00, 160.68it/s]

✅ Video saved successfully at: ../sample_output.mp4
